# Lesson 10: logistic regression on recession warnings

1. How does logistic regression turn the term spread into a recession probability?
2. How are a model's mistakes counted at one threshold?
3. Where should the threshold sit when a missed recession and a false alarm cost different amounts?
4. How are two models compared across every threshold?

Monthly US data, April 1953 to September 2026: the term spread, the 10-year Treasury yield minus the 3-month bill rate, and the NBER recession months. A month is a pre-recession month when the month 12 months later is a recession month; the label ends in September 2024. Training months run to December 1999, test months from January 2000.

Treasury yields and recession dates: Federal Reserve Board H.15 and NBER via FRED, Federal Reserve Bank of St. Louis. Snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/data-science-msc/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot: "us-recession-monthly"."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err


import numpy as np
import matplotlib.pyplot as plt

## 1. Term spread and recessions, 1953 to 2026

The split is by date: every training month lies before every test month, since neighbouring months share most of their spread.

In [ ]:
months = load("us-recession-monthly")
train = months[months["split"] == "train"]
test = months[months["split"] == "test"]
print(months.shape, train.shape, test.shape)
assert train["month"].max() < test["month"].min()  # every training month lies before every test month
y_train = train["recession_in_12m"].astype(int).to_numpy()
y_test = test["recession_in_12m"].astype(int).to_numpy()
print("pre-recession months: training", y_train.sum(), "of", len(y_train), "| test", y_test.sum(), "of", len(y_test))
print(f"share of pre-recession months: training {y_train.mean():.3f}, test {y_test.mean():.3f}")
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(pd.to_datetime(months["month"]), months["spread"], lw=0.8, color="black")
ax.fill_between(pd.to_datetime(months["month"]), -3, 5, where=months["usrec"] == 1, color="tab:red", alpha=0.2, lw=0)
ax.axhline(0, color="grey", ls="--")
ax.set_ylabel("10-year minus 3-month yield (points)")
ax.set_title("Term spread; red: NBER recession months")
plt.show()

## 2. From the spread to a probability

The score $z = b + w \cdot \text{spread}$ adds the weighted spread to a base value, and the sigmoid $\sigma(z) = 1/(1 + e^{-z})$ turns it into a probability between 0 and 1. The six months of the slides use the fitted weights rounded to one decimal, $b = -0.7$ and $w = -1.4$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

six = months.set_index("month").loc[["2000-11", "2007-03", "2011-06", "2017-06", "2019-04", "2023-06"],
                                     ["spread", "recession_in_12m"]].copy()
six["z"] = -0.7 - 1.4 * six["spread"]  # the fitted weights rounded to one decimal
six["p"] = sigmoid(six["z"]).round(3)
for month, row in six.iterrows():
    print(f"{month}  spread {row['spread']:5.2f}  z {row['z']:5.2f}  p {row['p']:.3f}  pre-recession {int(row['recession_in_12m'])}")
zs = np.linspace(-5, 3, 200)
plt.plot(zs, sigmoid(zs), color="black")
plt.scatter(six["z"], six["p"], c=np.where(six["recession_in_12m"] == 1, "tab:red", "tab:blue"))
plt.axhline(0.5, ls="--", c="grey")
plt.xlabel("score z")
plt.ylabel("recession probability p")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression


def fit(cols, **options):
    """Logistic regression without penalty, so a weight is per percentage point of its feature."""
    model = LogisticRegression(penalty=None, solver="newton-cholesky", max_iter=5000, **options)
    return model.fit(train[cols], y_train)


model_a = fit(["spread"])
model_b = fit(["spread", "tbill_change_12m"])
print(f"A: intercept {model_a.intercept_[0]:.3f}, spread weight {model_a.coef_[0][0]:.3f}, odds factor {np.exp(model_a.coef_[0][0]):.2f}")
weights = pd.Series(model_b.coef_[0], index=["spread", "tbill_change_12m"])
print(pd.DataFrame({"weight per point": weights.round(2), "odds factor": np.exp(weights).round(2)}))
weights.plot.barh(color=np.where(weights > 0, "tab:blue", "tab:red"))
plt.xlabel("weight per percentage point (model B)")
plt.show()

## 3. Counting mistakes at one threshold

A month gets a warning when $p \ge t$. The confusion matrix counts hits (TP), misses (FN), false alarms (FP) and calm months left calm (TN).

In [ ]:
def counts(y, p, t):
    """TP, FN, FP, TN when a month with p >= t gets a warning."""
    pred = p >= t
    return (int((pred & (y == 1)).sum()), int((~pred & (y == 1)).sum()),
            int((pred & (y == 0)).sum()), int((~pred & (y == 0)).sum()))

p_a = model_a.predict_proba(test[["spread"]])[:, 1]
for t in (0.5, 0.2):
    tp, fn, fp, tn = counts(y_test, p_a, t)
    print(f"threshold {t}: TP {tp}  FN {fn}  FP {fp}  TN {tn}")
    print(f"  accuracy {(tp + tn) / len(y_test):.3f}  recall {tp / (tp + fn):.3f}  precision {tp / (tp + fp):.3f}")
from sklearn.metrics import confusion_matrix, precision_score, recall_score
print(confusion_matrix(y_test, p_a >= 0.2), precision_score(y_test, p_a >= 0.2), recall_score(y_test, p_a >= 0.2))
alarms = test["month"][(p_a >= 0.5) & (y_test == 0)]
print("false alarms at 0.5:", len(alarms), "from", alarms.min(), "to", alarms.max())

In [ ]:
tp, fn, fp, tn = counts(y_test, np.zeros(len(y_test)), 0.5)
print(f"never-warn rule: accuracy {(tp + tn) / len(y_test):.3f}  recall {tp / (tp + fn):.0f}")
print("precision: TP + FP =", tp + fp, "so the ratio has no value")

## 4. A threshold from costs

A missed recession costs 24 and a false alarm 6; both costs are stylised. A warning pays when $24p > 6(1 - p)$, above $p = 0.2$. The threshold is picked on the 561 training months, the months that fitted model A, and its cost is then measured on the 297 test months.

In [ ]:
COST_MISS, COST_ALARM = 24, 6  # stylised


def total_cost(y, p, t, miss=COST_MISS):
    """Cost of the misses and false alarms when a month with p >= t gets a warning."""
    tp, fn, fp, tn = counts(y, p, t)
    return fn * miss + fp * COST_ALARM


p_a_train = model_a.predict_proba(train[["spread"]])[:, 1]
ts = np.round(np.arange(0.05, 0.96, 0.01), 2)
cost_train = [total_cost(y_train, p_a_train, t) for t in ts]
cost_test = [total_cost(y_test, p_a, t) for t in ts]
best = ts[int(np.argmin(cost_train))]  # picked on the training months
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, cost, title in zip(axes, (cost_train, cost_test), ("561 training months", "297 test months")):
    ax.plot(ts, cost, color="black")
    ax.axvline(0.2, ls="--", c="grey")
    ax.plot(best, cost[list(ts).index(best)], "o", color="tab:orange")
    ax.set_xlabel("threshold")
    ax.set_ylabel("cost")
    ax.set_title(title)
plt.show()
for t in (0.5, 0.2, best):
    tp, fn, fp, tn = counts(y_test, p_a, t)
    print(f"threshold {t:.2f}: training cost {cost_train[list(ts).index(t)]}, test cost {cost_test[list(ts).index(t)]}"
          f"  (test TP {tp}  FN {fn}  FP {fp}  TN {tn})")
print(f"break-even {COST_ALARM / (COST_ALARM + COST_MISS):.2f} | picked on the training months {best:.2f} | never warn on the test months {y_test.sum() * COST_MISS}")

## 5. Comparing two models over all thresholds

The ROC curve plots true positive rate against false positive rate for every threshold. The AUC is the share of (pre-recession, calm) pairs in which the pre-recession month has the higher probability, a tie counting half.

In [ ]:
def auc_by_pairs(p, y):
    """Share of (pre-recession, calm) pairs in which the pre-recession month has the higher p; a tie counts half."""
    pos, neg = p[y == 1], p[y == 0]
    return float(np.mean(pos[:, None] > neg[None, :]) + 0.5 * np.mean(pos[:, None] == neg[None, :]))

print(f"six months: {auc_by_pairs(six['p'].to_numpy(), six['recession_in_12m'].to_numpy()):.3f}")
from sklearn.metrics import roc_auc_score, roc_curve
p_b = model_b.predict_proba(test[["spread", "tbill_change_12m"]])[:, 1]
for name, model, cols, p in [("A: spread", model_a, ["spread"], p_a),
                             ("B: spread and bill-rate change", model_b, ["spread", "tbill_change_12m"], p_b)]:
    auc_train = roc_auc_score(y_train, model.predict_proba(train[cols])[:, 1])
    fpr, tpr, _ = roc_curve(y_test, p)
    print(f"{name}: AUC train {auc_train:.3f}, test {roc_auc_score(y_test, p):.3f}, TPR at FPR 0.2 {tpr[fpr <= 0.2].max():.3f}")
    plt.plot(fpr, tpr, label=f"{name}: AUC train {auc_train:.3f}, test {roc_auc_score(y_test, p):.3f}")
plt.plot([0, 1], [0, 1], c="grey", ls="--")
plt.xlabel("false positive rate")
plt.ylabel("true positive rate")
plt.legend()
plt.show()

## Exercises

**E1.** A missed recession costs 54, a false alarm still 6. Where does the cheapest threshold on the training months move, and what does it cost on the test months?

**E2.** Fit model A with `class_weight="balanced"`, which weighs the 77 pre-recession training months as heavily as the 484 calm ones. Compare precision and recall at threshold 0.5 with section 3.

**E3.** Compute the test AUC of model A by counting all 28 × 269 pairs, and compare it with `roc_auc_score`.

In [ ]:
# E1: a missed recession costs 54, a false alarm still 6
cost54_train = [total_cost(y_train, p_a_train, t, miss=54) for t in ts]
best54 = ts[int(np.argmin(cost54_train))]
print(f"picked on the training months when a miss costs 54: {best54:.2f}  when it costs 24: {best:.2f}")
print(f"test cost at {best54:.2f} with a miss at 54: {total_cost(y_test, p_a, best54, miss=54)}")

In [ ]:
# E2: balanced class weights
balanced = fit(["spread"], class_weight="balanced")
tp, fn, fp, tn = counts(y_test, balanced.predict_proba(test[["spread"]])[:, 1], 0.5)
print(f"balanced at 0.5: precision {tp / (tp + fp):.3f}  recall {tp / (tp + fn):.3f}")

In [ ]:
# E3: AUC by counting all 28 x 269 test pairs
print(f"{auc_by_pairs(p_a, y_test):.3f} {roc_auc_score(y_test, p_a):.3f}")

## Logistic regression gives the probability, and the costs of two mistakes pick the threshold

1. A weighted sum of the features gives a score $z$, and the sigmoid $1/(1 + e^{-z})$ turns it into a probability between 0 and 1. A weight $w$ multiplies the odds of a recession by $e^{w}$ per point of its feature; the spread's weight $-1.44$ gives 0.24.
2. A threshold turns each probability into a warning or none, and the confusion matrix counts hits, misses, false alarms and calm months left calm. Accuracy, precision and recall are ratios of its cells.
3. A warning pays when the expected cost of silence, $24p$, exceeds that of a warning, $6(1 - p)$, which holds above $p = 0.2$. The most accurate rule, never warning, misses all 28 pre-recession months.
4. The ROC curve plots true positive rate against false positive rate for every threshold, and the AUC is the share of (pre-recession, calm) month pairs the model ranks correctly. The AUC ranks the models; the costs still pick the threshold.

## Appendix (optional): fitting by gradient descent

scikit-learn finds the weights that minimise the cross-entropy of the training labels. Gradient descent approaches the same minimum in small steps: 300 steps of size 0.5 on the standardised spread bring the weights near model A's $-0.694$ and $-1.440$.

In [ ]:
x = (train["spread"] - train["spread"].mean()) / train["spread"].std()
X = np.column_stack([np.ones(len(x)), x])
w = np.zeros(2)
loss = []
for step in range(300):
    p = 1 / (1 + np.exp(-X @ w))
    loss.append(-np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p)))
    w -= 0.5 * X.T @ (p - y_train) / len(y_train)
plt.plot(loss)
plt.xlabel("step")
plt.ylabel("cross-entropy on the training months")
plt.show()
print("weights on the standardised spread", w.round(3))
mean, sd = train["spread"].mean(), train["spread"].std()
print(f"on the spread in points: intercept {w[0] - w[1] * mean / sd:.3f}, weight {w[1] / sd:.3f}")

## Newer data (optional)

The cells below fetch the live series from FRED. Their numbers differ from the slides, which use the snapshot of 30 September 2026.

In [ ]:
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="


def fred(*ids):
    """Live FRED series, e.g. fred("GS10", "TB3MS", "USREC"); "." marks a missing value."""
    return pd.read_csv(FRED + ",".join(ids), na_values=["."], parse_dates=["observation_date"])


live = fred("GS10", "TB3MS").dropna().tail(12).copy()
live["spread"] = live["GS10"] - live["TB3MS"]
live["p_recession_12m"] = model_a.predict_proba(live[["spread"]])[:, 1].round(3)
print(live[["observation_date", "spread", "p_recession_12m"]])